# Phase 2: bi-encoder fine-tuning (Colab)

Runtime > Change runtime type > **T4 GPU**. This notebook only calls the repo's scripts; all logic lives in `src/finqa`.
Only the **dev** split is evaluated. Test is never touched here.

In [ ]:
!git clone https://github.com/IbrahimKhan25/financial-qa-retrieval.git
%cd financial-qa-retrieval

In [ ]:
!python --version  # the project requires Python 3.11 or newer
# torch and transformers come preinstalled on Colab; the results JSON records the torch version used.
!pip install -q -e . bm25s scipy mlflow
import torch
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
!python scripts/download_data.py

## Row B: in-batch negatives only

In [ ]:
!python scripts/train_bi_encoder.py --config configs/bi_encoder_B_inbatch.yaml

## Row C: plus BM25 hard negatives

In [ ]:
!python scripts/mine_negatives.py --source bm25 --out data/negatives/bm25.json
!python scripts/train_bi_encoder.py --config configs/bi_encoder_C_bm25.yaml

## Optional row C2: BM25 negatives with false negatives filtered
Drops any doc that is a positive for *any* train query. Run it to test whether false negatives were hurting row C; skip it to save time.


In [ ]:
!python scripts/mine_negatives.py --source bm25 --exclude-train-positives --out data/negatives/bm25_filtered.json
!python scripts/train_bi_encoder.py --config configs/bi_encoder_C2_bm25_filtered.yaml


## Row D: plus negatives mined from model C
Starts again from the original MiniLM (not from C), so any change comes from the negatives, not from extra training.

In [ ]:
!python scripts/mine_negatives.py --source dense --checkpoint checkpoints/bi_encoder_C_bm25 --out data/negatives/mined_C.json
!python scripts/train_bi_encoder.py --config configs/bi_encoder_D_mined.yaml

## Download the results (copy them into the repo's `results/` folder)

In [ ]:
!zip -j results_phase2.zip results/train_*.json
from google.colab import files
files.download("results_phase2.zip")

## Phase 4 (reranker)
Moved to its own notebook: `notebooks/colab_train_reranker.ipynb`.